## 🎯 Learning Objectives
* Understand the distinct roles and responsibilities of retriever, responder, and escalation agents within an AutoGen crew.
* Learn to define and configure specialized agents using AutoGen's `AssistantAgent` and `UserProxyAgent` for a multi-agent RAG workflow.
* Implement a basic multi-agent conversation flow simulating an e-commerce catalog search and query routing.
* Analyze the interaction patterns and benefits of an agentic approach for handling complex or ambiguous user queries in a RAG system.


## AutoGen Agent Crew: Retriever, Responder, Escalation Agent

In the realm of advanced AI systems, particularly those involving Retrieval Augmented Generation (RAG), a single, monolithic agent often falls short when faced with complex, nuanced, or ambiguous queries. This is where the power of an **agent crew** shines. By distributing specialized tasks among multiple agents, we can build more robust, accurate, and intelligent systems. For our e-commerce RAG project, we'll adopt a highly effective pattern: the **Retriever, Responder, and Escalation Agent** crew.

Imagine a highly efficient customer service department, but staffed by AI agents:

1.  **The Retriever Agent (The Information Specialist):**
    *   **Role:** This agent is the first line of defense. Its primary job is to quickly and efficiently search through a vast knowledge base (our e-commerce product catalog, FAQs, user manuals, etc.) to find relevant information snippets. It doesn't try to answer the question directly, but rather to *retrieve* the most pertinent data.
    *   **Analogy:** Think of a librarian or a front-line support agent with instant access to a comprehensive CRM. They excel at finding facts and documents but might not be the best at synthesizing a complex answer.
    *   **In RAG:** This agent would typically interact with a vector database, search API, or a structured knowledge graph to fetch product details, specifications, reviews, or policy information based on the user's query.

2.  **The Responder Agent (The Communicator/Synthesizer):**
    *   **Role:** Once the Retriever Agent has gathered potential information, the Responder Agent takes over. Its task is to synthesize this raw data into a coherent, concise, and user-friendly answer. It's responsible for crafting the final response that the user sees, ensuring it directly addresses the query while being easy to understand.
    *   **Analogy:** This is like the experienced customer service representative who takes the information found by the front-line agent and crafts a polite, comprehensive email or verbal response to the customer.
    *   **In RAG:** This agent processes the retrieved documents, extracts key insights, and generates a natural language response, potentially comparing products, explaining features, or summarizing policies.

3.  **The Escalation Agent (The Expert/Problem Solver):**
    *   **Role:** This agent is the safety net. It steps in when the Retriever and Responder agents are unable to provide a satisfactory answer. This could be due to insufficient information, an ambiguous query, a request that requires deeper analysis, or a query that falls outside the scope of the initial knowledge base. The Escalation Agent might ask clarifying questions, suggest alternative approaches, or even indicate that the system cannot fulfill the request.
    *   **Analogy:** This is the supervisor, the technical expert, or the specialized department that gets involved when the standard support channels can't resolve an issue. They have a broader understanding or specific tools to handle complex cases.
    *   **In RAG:** This agent prevents 


In [ ]:
import autogen
import os

# --- Configuration --- 
# In a real-world scenario, these would be loaded from environment variables or a config file.
# For 2026, we assume robust local LLM inference or highly optimized cloud endpoints.
# We'll use a placeholder for demonstration.

# Ensure you have your API key set up. For local models, this might be 'ollama' or 'vllm' endpoint.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY" # Or your local LLM endpoint key

# Define LLM configuration for all agents
# In 2026, we'd likely use a mix of specialized smaller models and larger general models.
# For this example, we'll use a generic 'gpt-4o' or a local equivalent.
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o", # Or a local model like "ollama/llama3" or "vllm/mixtral"
            "api_key": os.environ.get("OPENAI_API_KEY", "sk-no-key-needed"), # Placeholder if using local LLM
            "base_url": os.environ.get("OPENAI_API_BASE", "https://api.openai.com/v1") # Or your local LLM endpoint
        }
    ],
    "temperature": 0.1, # Keep it low for factual retrieval and synthesis
    "timeout": 120
}

# --- Simulate an E-commerce Product Catalog (Knowledge Base) ---
# In a real RAG system, this would be a vector database, search index, etc.
product_catalog = {
    "smart_speaker_x": {
        "name": "Smart Speaker X",
        "features": "Voice assistant, premium audio, smart home hub, Wi-Fi 6, Bluetooth 5.2, privacy controls.",
        "price": "$199",
        "brand": "AudioTech",
        "category": "Smart Home",
        "compatibility": "Works with all major smart home ecosystems."
    },
    "wireless_earbuds_y": {
        "name": "Wireless Earbuds Y",
        "features": "Active Noise Cancellation, 30-hour battery life, IPX7 waterproof, touch controls, spatial audio.",
        "price": "$149",
        "brand": "SoundFlow",
        "category": "Audio",
        "compatibility": "Universal Bluetooth compatibility."
    },
    "gaming_laptop_z": {
        "name": "Gaming Laptop Z",
        "features": "16-inch QHD display, NVIDIA RTX 5080 GPU, Intel Core i9 (16th Gen), 32GB RAM, 1TB SSD, RGB keyboard.",
        "price": "$2499",
        "brand": "GamerPro",
        "category": "Computers",
        "compatibility": "Windows 12."
    },
    "smart_speaker_a": {
        "name": "Smart Speaker A",
        "features": "Compact design, basic voice assistant, good for small rooms, Bluetooth 5.0.",
        "price": "$49",
        "brand": "EchoLite",
        "category": "Smart Home",
        "compatibility": "Works with EchoLite ecosystem."
    }
}

# --- Define the Agents ---

# 1. Retriever Agent
# This agent's job is to 'search' the product_catalog. 
# In a real system, it would call a RAG tool.
retriever_agent = autogen.AssistantAgent(
    name="Retriever",
    system_message=(
        "You are a highly efficient product information retriever. Your sole purpose is to search "
        "the provided `product_catalog` for relevant information based on the user's query. "
        "Do NOT try to answer the question directly. Instead, extract and present ONLY the raw, "
        "relevant product details in a structured format (e.g., JSON or bullet points). "
        "If you find multiple relevant products, provide details for all of them. "
        "If no relevant information is found, state 'No relevant product information found.'"
        f"Available products and their keys: {list(product_catalog.keys())}."
        "You have access to the `product_catalog` dictionary directly. "
        "To retrieve information, you can iterate through `product_catalog.values()` and check if keywords from the query are in the product's name or features. "
        "Present the found details clearly to the Responder agent."
    ),
    llm_config=llm_config,
    is_termination_msg=lambda x: "No relevant product information found." in x.get("content", "") or "Here is the retrieved information" in x.get("content", "")
)

# 2. Responder Agent
# This agent synthesizes the retrieved information into a user-friendly answer.
responder_agent = autogen.AssistantAgent(
    name="Responder",
    system_message=(
        "You are a friendly and helpful e-commerce customer service agent. "
        "Your task is to take the information provided by the Retriever agent and "
        "craft a clear, concise, and comprehensive answer to the user's original query. "
        "If the Retriever found no information, politely state that you couldn't find details. "
        "Always aim to provide a complete answer based *only* on the retrieved data. "
        "Do not invent information. Once you have provided a complete answer, say 'TERMINATE'."
    ),
    llm_config=llm_config,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").upper()
)

# 3. Escalation Agent
# This agent steps in if the initial agents can't resolve the query or if it's complex.
escalation_agent = autogen.AssistantAgent(
    name="EscalationAgent",
    system_message=(
        "You are a specialized e-commerce expert. You are invoked when the Retriever and Responder "
        "agents are unable to fully address a user's query, either due to lack of specific information, "
        "ambiguity, or the need for deeper analysis (e.g., comparison, troubleshooting). "
        "Your role is to either: "
        "1. Ask clarifying questions to the user to better understand their needs. "
        "2. Suggest alternative products or categories if the initial search was too narrow. "
        "3. Provide general advice if specific product details are missing but the query can still be addressed broadly. "
        "4. If the query is truly unanswerable with current resources, politely state so and suggest next steps (e.g., contact human support). "
        "Always aim to move the conversation forward or provide a definitive conclusion. "
        "Once you have provided a comprehensive response or clarification, say 'TERMINATE'."
    ),
    llm_config=llm_config,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").upper()
)

# User Proxy Agent to represent the human user and facilitate conversation
user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER", # Set to "ALWAYS" for interactive debugging
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").upper(),
    code_execution_config=False, # No code execution for this example
)

# --- Define the Group Chat and Manager ---
# The manager orchestrates the conversation flow.

groupchat = autogen.GroupChat(
    agents=[user_proxy, retriever_agent, responder_agent, escalation_agent],
    messages=[],
    max_round=15, # Limit conversation rounds to prevent infinite loops
    speaker_selection_method="auto", # Auto-selects the next speaker based on context
    allow_repeat_speaker=False # Prevent an agent from speaking twice in a row
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

# --- Initiate Conversations ---
print("\n--- Conversation 1: Simple Product Query ---")
user_proxy.initiate_chat(
    manager,
    message="Tell me about the features of Smart Speaker X."
)

print("\n--- Conversation 2: Comparison Query (Requires more synthesis) ---")
user_proxy.initiate_chat(
    manager,
    message="Compare the Smart Speaker X and Smart Speaker A. Which one is better for a large living room?"
)

print("\n--- Conversation 3: Ambiguous/Missing Info Query (Should trigger Escalation) ---")
user_proxy.initiate_chat(
    manager,
    message="I need a new gadget for my kitchen. What do you recommend?"
)

print("\n--- Conversation 4: Non-existent Product Query (Should trigger Escalation/No Info) ---")
user_proxy.initiate_chat(
    manager,
    message="What are the specifications of the Quantum Leap VR Headset?"
)


### Interpreting the Code Output and Workflow

When you run the code, you'll observe a dynamic conversation unfold between the agents, orchestrated by the `GroupChatManager`. Here's what to look for and how to interpret the interactions:

1.  **Conversation Flow:**
    *   **User Proxy initiates:** The `User` (our `UserProxyAgent`) sends the initial query to the `manager`.
    *   **Retriever's Turn:** The `manager` will typically direct the query first to the `Retriever` agent, as its system message primes it to search for information. You'll see the `Retriever` attempting to find relevant data from the `product_catalog` (our simulated knowledge base).
    *   **Responder's Turn:** If the `Retriever` successfully finds information, it will pass this raw data to the `Responder`. The `Responder` then processes this information and formulates a user-friendly answer. Its `is_termination_msg` will signal `TERMINATE` once it's done.
    *   **Escalation's Turn:** The `EscalationAgent` comes into play when the `Retriever` finds no information, or when the `Responder` indicates it cannot fully answer the query based on the retrieved data (though in our simple example, the `Responder` might just state it couldn't find info, and the `EscalationAgent` then takes over to provide a more helpful next step).

    You'll see messages like `Retriever (to chat_manager)` followed by its output, then `Responder (to chat_manager)` and its response, and so on. This chain of communication demonstrates the specialized roles in action.

2.  **Successful vs. Escalated Queries:**
    *   **Simple Query (e.g., "Smart Speaker X features"):** The `Retriever` quickly finds the details, the `Responder` synthesizes them into a clear answer, and the conversation terminates successfully.
    *   **Comparison Query (e.g., "Compare Smart Speaker X and A"):** The `Retriever` should find details for both. The `Responder` then needs to perform a more complex synthesis, comparing features and potentially making a recommendation based on the user's implied need ("better for a large living room"). This showcases the `Responder`'s ability to go beyond mere summarization.
    *   **Ambiguous/Missing Info Query (e.g., "gadget for my kitchen"):** The `Retriever` will likely find no direct matches. The `Responder` will acknowledge this. This is where the `EscalationAgent` should activate, asking clarifying questions ("What kind of kitchen gadget are you looking for? What's your budget?") or suggesting categories, demonstrating its role in guiding the user or handling unspecific requests.
    *   **Non-existent Product Query (e.g., "Quantum Leap VR Headset"):** Similar to the ambiguous query, the `Retriever` will find nothing. The `EscalationAgent` will then step in to inform the user that the product isn't in the catalog and perhaps suggest browsing related categories.

### Performance Trade-offs and Use Cases

**Performance Trade-offs:**

*   **Pros:**
    *   **Enhanced Accuracy and Robustness:** By delegating tasks, each agent can be highly optimized for its specific role, leading to more precise retrieval and more accurate, less hallucinatory responses.
    *   **Improved Handling of Complexity:** Complex queries are broken down into manageable sub-tasks, allowing the system to address nuanced requests that a single agent might struggle with.
    *   **Modularity and Maintainability:** Each agent's system message and tools can be independently updated and refined, making the system easier to debug, extend, and adapt to new requirements.
    *   **Better User Experience:** Escalation agents prevent dead ends, offering guidance or alternatives instead of simply stating 


### Resources

*   **AutoGen Documentation:** The official source for understanding AutoGen's architecture, agent types, and group chat capabilities.
    *   [AutoGen GitHub Repository](https://github.com/microsoft/autogen)
    *   [AutoGen Documentation](https://microsoft.github.io/autogen/)
*   **Microsoft Research on AutoGen:** Dive deeper into the research and principles behind multi-agent conversations.
    *   [AutoGen: Enabling Next-Gen LLM Applications with Multi-Agent Conversation Framework](https://www.microsoft.com/en-us/research/blog/autogen-enabling-next-gen-llm-applications-with-multi-agent-conversation-framework/)
*   **Retrieval Augmented Generation (RAG) Overview:** Understand the core concept that these agents are designed to enhance.
    *   [Retrieval Augmented Generation (RAG) Explained](https://www.ibm.com/topics/retrieval-augmented-generation)
    *   [Hugging Face Course on RAG](https://huggingface.co/learn/nlp-course/chapter7/6?fw=pt)
*   **LLM Providers (for 2026 context):**
    *   [OpenAI API Documentation](https://platform.openai.com/docs/)
    *   [Google AI Studio / Gemini API](https://ai.google.dev/)
    *   [Hugging Face Inference Endpoints](https://huggingface.co/inference-endpoints)
    *   [Ollama (for local LLMs)](https://ollama.com/)
    *   [vLLM (for high-throughput LLM serving)](https://vllm.ai/)
